
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [6]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [7]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

,prompt_text,prompt_title
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [8]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    # TODO: yield slices of items of length batch_size
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def summarize_with_t5(
    texts: List[str],
    model_name: str = "t5-small",
    batch_size: int = 4,
    max_new_tokens: int = 32
):
    # TODO: load tokenizer/model, send to device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    summaries = []

    # TODO: tokenize with prefix, generate, decode
    for batch in batch_generator(texts, batch_size):
        prefixed_batch = ["summarize: " + text for text in batch]

        inputs = tokenizer(
            prefixed_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

        decoded = tokenizer.batch_decode(
            outputs,
            skip_special_tokens=True
        )

        summaries.extend(decoded)

        # TODO: clear caches between batches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return summaries


# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False

if RUN_T5:
    train_summaries_t5 = summarize_with_t5(
        train_df["prompt_text"].tolist(),
        model_name="t5-small",
        batch_size=2
    )

    display(pd.DataFrame({
        "prompt_text": train_df["prompt_text"],
        "reference_summary": train_df["prompt_title"],
        "t5_small_summary": train_summaries_t5
    }).head())

else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")

Skipping T5 generation for speed. Set RUN_T5=True to execute.



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [9]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")


Accuracy skipped (no predictions).



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [15]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load("rouge")

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    # TODO: normalize preds/refs; call rouge.compute
    norm_preds = [normalize_text(p) for p in preds]
    norm_refs = [normalize_text(r) for r in refs]

    scores = rouge.compute(
        predictions=norm_preds,
        references=norm_refs,
        use_stemmer=True
    )

    return scores

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))

ROUGE sanity check (fill function first):
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.6666666666666666), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


In [16]:
# Part VI. Understanding ROUGE scores

print("1. Exact match vs. empty prediction")
preds = ["The cat sat on the mat.", ""]
refs = ["The cat sat on the mat.", "The cat sat on the mat."]
print(compute_rouge_score(preds, refs))


print("\n2. Effect of stemming: running vs. run")

preds = ["The runner was running fast."]
refs = ["The runner run fast."]

# With stemming
print("With stemming:")
print(compute_rouge_score(preds, refs))

# Without stemming
norm_preds = [normalize_text(p) for p in preds]
norm_refs = [normalize_text(r) for r in refs]

scores_no_stemmer = rouge.compute(
    predictions=norm_preds,
    references=norm_refs,
    use_stemmer=False
)

print("Without stemming:")
print(scores_no_stemmer)


print("\n3. N-gram overlap: partial overlap")

preds = ["The cat sat on the mat."]
refs = ["The cat slept on the sofa."]
print(compute_rouge_score(preds, refs))


print("\n4. Symmetry: swap preds and refs")

preds = ["The cat sat on the mat."]
refs = ["The cat sat."]

print("Original:")
print(compute_rouge_score(preds, refs))

print("Swapped:")
print(compute_rouge_score(refs, preds))

1. Exact match vs. empty prediction
{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.5), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}

2. Effect of stemming: running vs. run
With stemming:
{'rouge1': np.float64(0.888888888888889), 'rouge2': np.float64(0.5714285714285715), 'rougeL': np.float64(0.888888888888889), 'rougeLsum': np.float64(0.888888888888889)}
Without stemming:
{'rouge1': np.float64(0.6666666666666665), 'rouge2': np.float64(0.28571428571428575), 'rougeL': np.float64(0.6666666666666665), 'rougeLsum': np.float64(0.6666666666666665)}

3. N-gram overlap: partial overlap
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.4000000000000001), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}

4. Symmetry: swap preds and refs
Original:
{'rouge1': np.float64(0.6666666666666666), 'rouge2': np.float64(0.5714285714285715), 'rougeL': np.float64(0.6666666666666666), 'rougeLsum': np.float64(0.6666666666666666)}
Swapped:
{'ro

ROUGE behaves as expected.

For exact match vs. empty prediction, the exact match gets a high ROUGE score because the generated text is identical to the reference. The empty prediction gets zero because it has no overlapping words with the reference. This shows that ROUGE strongly depends on word overlap.

For stemming, the scores are higher when stemming is used. For example, “running” and “run” are different word forms, but stemming reduces them to a similar root. Without stemming, ROUGE treats them more literally, so the score is lower.

For n-gram overlap, ROUGE-1 is higher than ROUGE-2 in the partial-overlap example. This is because ROUGE-1 checks individual word overlap, while ROUGE-2 checks pairs of consecutive words. It is easier for single words to overlap than for full two-word phrases to match.

For symmetry, swapping predictions and references gives similar or identical scores in this small example. However, in a real summarization task, the model-generated summary should still be treated as the prediction and the human-written summary should be treated as the reference.


### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [19]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(
    texts: List[str],
    model_name: str = "gpt2",
    batch_size: int = 2,
    max_new_tokens: int = 32
):
    # TODO: implement simple TL;DR generation
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    summaries = []

    for batch in batch_generator(texts, batch_size):
        prompts = [
            "Summarize the following text.\n\n"
            + text[:1500]
            + "\n\nTL;DR:"
            for text in batch
        ]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

        for output_ids, input_ids in zip(outputs, inputs["input_ids"]):
            generated_ids = output_ids[len(input_ids):]
            summary = tokenizer.decode(
                generated_ids,
                skip_special_tokens=True
            ).strip()

            summaries.append(summary)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    return summaries


def compute_rouge_per_row(
    df: pd.DataFrame,
    pred_col: str,
    ref_col: str = "prompt_title"
):
    # TODO: for each row, compute ROUGE-L or ROUGE-1/2 and store results
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for _, row in df.iterrows():
        scores = compute_rouge_score(
            [row[pred_col]],
            [row[ref_col]]
        )

        rouge1_scores.append(scores["rouge1"])
        rouge2_scores.append(scores["rouge2"])
        rougeL_scores.append(scores["rougeL"])

    df[f"{pred_col}_rouge1"] = rouge1_scores
    df[f"{pred_col}_rouge2"] = rouge2_scores
    df[f"{pred_col}_rougeL"] = rougeL_scores

    return df


RUN_COMPARE = True

if RUN_COMPARE:
    compare_df = train_df.head(10).copy()

    compare_df["t5_small_summary"] = summarize_with_t5(
        compare_df["prompt_text"].tolist(),
        model_name="t5-small",
        batch_size=2,
        max_new_tokens=32
    )

    compare_df["t5_base_summary"] = summarize_with_t5(
        compare_df["prompt_text"].tolist(),
        model_name="t5-base",
        batch_size=1,
        max_new_tokens=32
    )

    compare_df["gpt2_summary"] = summarize_with_gpt2(
        compare_df["prompt_text"].tolist(),
        model_name="gpt2",
        batch_size=2,
        max_new_tokens=32
    )

    compare_df = compute_rouge_per_row(compare_df, "t5_small_summary")
    compare_df = compute_rouge_per_row(compare_df, "t5_base_summary")
    compare_df = compute_rouge_per_row(compare_df, "gpt2_summary")

    display(compare_df[[
        "prompt_title",
        "t5_small_summary",
        "t5_small_summary_rouge1",
        "t5_small_summary_rouge2",
        "t5_small_summary_rougeL",
        "t5_base_summary",
        "t5_base_summary_rouge1",
        "t5_base_summary_rouge2",
        "t5_base_summary_rougeL",
        "gpt2_summary",
        "gpt2_summary_rouge1",
        "gpt2_summary_rouge2",
        "gpt2_summary_rougeL"
    ]])

else:
    print("Comparison skipped. Set RUN_COMPARE=True to execute.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

,prompt_title,t5_small_summary,t5_small_summary_rouge1,t5_small_summary_rouge2,t5_small_summary_rougeL,t5_base_summary,t5_base_summary_rouge1,t5_base_summary_rouge2,t5_base_summary_rougeL,gpt2_summary,gpt2_summary_rouge1,gpt2_summary_rouge2,gpt2_summary_rougeL
0,Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,0.222222,0.131148,0.222222,championship leader Lewis Hamilton spun out of...,0.548387,0.300000,0.516129,The world championship race is over. The world...,0.061538,0.000000,0.061538
1,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,0.147059,0.000000,0.117647,Xinhua: some children who swallowed the beads ...,0.208955,0.123077,0.179104,The Aqua Dots toys contain a chemical that can...,0.166667,0.028571,0.111111
2,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,0.424242,0.156250,0.333333,Qualcomm was founded in 1985 by seven communic...,0.242424,0.031250,0.181818,Qualcomm is a global leader in the mobile tele...,0.298507,0.123077,0.208955
3,NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",0.237288,0.035088,0.203390,new: police arrest acting president of ex-Prim...,0.246154,0.000000,0.215385,Musharraf's actions are for the good of Pakist...,0.347826,0.208955,0.289855
4,Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,0.413793,0.142857,0.275862,third seed Julia vakulenko will face former wo...,0.500000,0.241379,0.300000,Julia Vakulenko is back in the WTA Tour and is...,0.372881,0.175439,0.237288
5,San Diego mayor declares state of emergency; W...,new: mayor says he received offers of aid from...,0.242424,0.031250,0.151515,landslide pulls earth from beneath three-lane ...,0.171429,0.000000,0.085714,The road was collapsing under the hillside. Th...,0.109589,0.000000,0.109589
6,NEW: Indictment: Man tried to pass nuclear fil...,sources say the classified materials were take...,0.083333,0.000000,0.055556,former contractor indicted on charges of steal...,0.187500,0.064516,0.187500,The FBI has been investigating Oakley for year...,0.135135,0.000000,0.108108
7,"NEW: Teen gunman is dead, Finnish police say ....",police say 18-year-old pekka Eric Auvinen shot...,0.253968,0.065574,0.190476,police: 18-year-old shooter identified as pekk...,0.070175,0.000000,0.070175,A 17-year-old student shot and killed two peop...,0.264706,0.000000,0.176471
8,President Bush to address the Veterans of Fore...,president will argue against pulling out from ...,0.426230,0.237288,0.360656,president bush to talk about why u.s. should w...,0.366667,0.206897,0.300000,The war in Vietnam was a price paid by million...,0.158730,0.000000,0.126984
9,Harry Potter star Daniel Radcliffe gets £20M f...,he says he has no plans to fritter cash away o...,0.354839,0.266667,0.322581,young actor says he has no plans to fritter hi...,0.444444,0.360656,0.380952,Harry Potter star Daniel Radcliffe has no plan...,0.412698,0.360656,0.412698



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [20]:
import pandas as pd

def compare_models(rouge_dict):
    # TODO: take {model_name: rouge_scores_dict} -> DataFrame with averages
    rows = []

    for model_name, scores in rouge_dict.items():
        rows.append({
            "model": model_name,
            "rouge1": scores["rouge1"],
            "rouge2": scores["rouge2"],
            "rougeL": scores["rougeL"],
            "rougeLsum": scores["rougeLsum"]
        })

    return pd.DataFrame(rows).sort_values(by="rougeL", ascending=False)


def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    # TODO: subset columns for side-by-side viewing
    cols = ["prompt_text", "prompt_title"] + pred_cols
    return df[cols]

In [21]:
if "compare_df" in locals():
    rouge_dict = {
        "t5-small": compute_rouge_score(
            compare_df["t5_small_summary"].tolist(),
            compare_df["prompt_title"].tolist()
        ),
        "t5-base": compute_rouge_score(
            compare_df["t5_base_summary"].tolist(),
            compare_df["prompt_title"].tolist()
        ),
        "gpt2": compute_rouge_score(
            compare_df["gpt2_summary"].tolist(),
            compare_df["prompt_title"].tolist()
        )
    }

    scores_table = compare_models(rouge_dict)
    display(scores_table)

    summaries_table = compare_models_summaries(
        compare_df,
        ["t5_small_summary", "t5_base_summary", "gpt2_summary"]
    )
    display(summaries_table.head())

else:
    print("Run Part VII with RUN_COMPARE=True first.")

,model,rouge1,rouge2,rougeL,rougeLsum
1,t5-base,0.297934,0.132962,0.238613,0.283636
0,t5-small,0.278632,0.106944,0.223736,0.253384
2,gpt2,0.230587,0.087302,0.185199,0.205828


,prompt_text,prompt_title,t5_small_summary,t5_base_summary,gpt2_summary
0,"SHANGHAI, China -- Championship leader Lewis H...",Lewis Hamilton fails to clinch world title aft...,championship leader Lewis Hamilton spun out of...,championship leader Lewis Hamilton spun out of...,The world championship race is over. The world...
1,(CNN) -- China has suspended exports of the Aq...,State-run news agency: China orders an investi...,china suspends exports of the toys contaminate...,Xinhua: some children who swallowed the beads ...,The Aqua Dots toys contain a chemical that can...
2,(CNN) -- The company was founded in 1985 by se...,The company has become a huge name in communic...,Qualcomm's patent portfolio includes approxima...,Qualcomm was founded in 1985 by seven communic...,Qualcomm is a global leader in the mobile tele...
3,"ISLAMABAD, Pakistan (CNN) -- Hours after decla...",NEW: President Musharraf orders troops to take...,"new: aides arrested, 10 arrested, aides arrest...",new: police arrest acting president of ex-Prim...,Musharraf's actions are for the good of Pakist...
4,"QUEBEC, Canada -- Third seed Julia Vakulenko w...",Julia Vakulenko has reached her first final on...,third seed Julia Vakulenko to face comeback qu...,third seed Julia vakulenko will face former wo...,Julia Vakulenko is back in the WTA Tour and is...



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.


The most informative metrics were ROUGE-1, ROUGE-2, and ROUGE-L. ROUGE-1 was useful because it showed general word overlap between the generated summary and the reference summary. ROUGE-2 was more strict because it measured two-word phrase overlap, so it helped show whether the model captured more exact wording and structure. ROUGE-L was especially useful because it measured longer sequence overlap, which is important for summarization quality.

Model size had a clear impact. t5-base performed better than t5-small on all ROUGE scores, which suggests that the larger model produced summaries closer to the human-written references. Qualitatively, t5-base also seemed more accurate and better aligned with the original articles. t5-small was still reasonable, but its summaries were slightly weaker. GPT-2 performed the worst because it is not specifically designed as a summarization model, even though it can generate fluent text.

Accuracy broke down because exact-match accuracy is too strict for summarization. A generated summary can be correct, clear, and useful even if it does not use exactly the same words as the reference summary. Since summarization is free-form text generation, there are many possible good answers. Therefore, exact string match usually gives a score close to zero and does not reflect real summary quality.

I would extend this evaluation with human evaluation. Human reviewers could rate each summary for factual accuracy, relevance, coherence, fluency, and whether it includes the most important information. I would also add adversarial probes, such as very long articles, articles with misleading details, contradictory information, or key information placed near the end. This would test whether the model truly understands the article instead of only matching surface-level word patterns.